# DDACS Dataset Tutorial

This tutorial demonstrates how to work with the [Deep Drawing and Cutting Simulations (DDACS) Dataset](https://ddacs.readthedocs.io) using the `ddacs` package.

## Installation

```bash
pip install ddacs
```

This installs the package with all core dependencies including visualization support.

For PyTorch integration (optional, see Section 6), install PyTorch first following the [official instructions](https://pytorch.org/get-started/locally/) for your CUDA version.

## Dataset Overview

### What is Deep Drawing?
Deep drawing is a manufacturing process where a sheet metal blank is formed into a hollow shape using a die and punch. The dataset captures the complete physics of this process including:

- **Material deformation**: How the metal sheet changes shape
- **Tool interaction**: Forces between forming tools and workpiece  
- **Process parameters**: Variables that control the outcome

### Data Hierarchy
```
Dataset
├── Simulations (32,071 total)
│   ├── Process Parameters (FC, MAT, STHK, BF, GEO, RAD)
│   └── Simulation Results (H5 files)
│       ├── OP10 (Forming Process)
│       │   ├── Components (blank, die, punch, binder)
│       │   └── Timesteps (0: initial → 2: max forming → 3: springback)
│       └── OP20 (Cutting Process)
│           ├── Component (blank)
│           └── Timesteps (0: initial → 1: springback)
```

### Key Components
- **Blank**: The workpiece being formed (most important for analysis)
- **Die**: Lower tool that provides the cavity shape
- **Punch**: Upper tool that pushes material into die
- **Binder**: Holds blank edges to control material flow

### Process Parameters
| Parameter | Range | Description |
|-----------|-------|-------------|
| FC | 0.05-0.15 | Friction coefficient |
| MAT | 0.9-1.1 | Material scaling factor |
| STHK | 0.95-1.0 mm | Sheet thickness |
| BF | 100k-500k N | Blank holder force |
| GEO | R/V/X | Geometry type |
| RAD | 30-150 mm | Characteristic radius |

In [ ]:
import numpy as np
import h5py
import pandas as pd
from pathlib import Path

import matplotlib.pyplot as plt

from ddacs import iter_ddacs, count_available_simulations
from ddacs.utils import (
    extract_point_cloud,
    extract_mesh,
    display_structure,
    extract_element_thickness,
    extract_point_springback,
)
from ddacs.visualization import (
    plot_mesh,
    plot_point_cloud,
    plot_vectors,
    plot_2d_projection,
    COMPONENT_COLORS,
    COMPONENT_NAMES,
)

In [ ]:
data_dir = Path("./../data")

# Count available simulations
num_simulations = count_available_simulations(data_dir)
print(f"Available simulations: {num_simulations}")

# Get first simulation for demonstration
sim_id, metadata, h5_path = next(iter_ddacs(data_dir, skip_missing=True))
print(f"\nSample simulation: {sim_id}")
print(f"File: {h5_path}")
print(f"Metadata: {metadata}")

## 2. Understanding the Metadata

The metadata contains the process and geometry parameters that define each simulation.

In [ ]:
# Load metadata directly from CSV
metadata_path = data_dir / "metadata.csv"
df = pd.read_csv(metadata_path)

print(f"Metadata structure:")
print(f"  Total simulations: {len(df)}")
print(f"  Columns: {list(df.columns)}")
print("  > Note: The 'ID' column maps to H5 filenames")

# Parameter descriptions
descriptions = {
    "GEO_R": "Rectangular geometry (1=yes, 0=no)",
    "GEO_V": "Concave geometry (1=yes, 0=no)",
    "GEO_X": "Convex geometry (1=yes, 0=no)",
    "RAD": "Characteristic radius [30-150 mm]",
    "MAT": "Material scaling factor [0.9-1.1]",
    "FC": "Friction coefficient [0.05-0.15]",
    "SHTK": "Sheet thickness [0.95-1.0 mm]",
    "BF": "Blank holder force [100,000-500,000 N]",
}

print(f"\nParameters (physical values):")
for key, desc in descriptions.items():
    print(f"  > {key:5} - {desc}")

print(f"\nSample metadata values:")
print(df.head())

## 3. Understanding the Simulation Data Structure

### HDF5 File Organization
Each simulation is stored as an HDF5 file containing:

```
simulation.h5
├── OP10/ (Forming process)
│   ├── blank/     (11k nodes, 4 timesteps) - The workpiece
│   ├── die/       (1k nodes, 3 timesteps)  - Lower tool
│   ├── punch/     (500 nodes, 3 timesteps) - Upper tool  
│   └── binder/    (100 nodes, 3 timesteps) - Edge clamp
└── OP20/ (Cutting process)
    └── blank/     (11k nodes, 2 timesteps) - Post-forming behavior
```

To display all information use [`display_structure(h5_path)`](https://ddacs.readthedocs.io/en/latest/api/utils/#ddacs.utils.display_structure) (see end of cell).

### Critical Timestep Understanding OP10
- **Tools have 3 timesteps** (0, 1, 2): They are removed after the deep drawing forming process.
- **Blank has 4 timesteps** (0, 1, 2, 3): Includes springback of forming process after tool removal.
- **Timestep 3**: Maximum forming depth (tools fully engaged)
- **Timestep 4**: Springback state (tools removed, blank partially recovers)

**Important**: Using `timestep=-1` gives different results for tools vs blank due to different timestep counts.

### Critical Timestep Understanding OP20
Operation 20 (`OP20`) depends in operation 10 and describes the cutting of the component after forming. In `OP20` **only** the `blank` component exists **Blank has 2 timesteps** (0, 1): Includes the cut `blank` component before and after springback.

In [ ]:
# Explore the HDF5 simulation structure
with h5py.File(h5_path, "r") as f:
    print("Operations:", list(f.keys()))

    print("\n ----------------------- \n")

    # OP10: Forming operation
    print("OP10 - Forming Process:")
    op10 = f["OP10"]
    print("  Components:", list(op10.keys()))

    # Analyze each component
    print("\nComponent Details:")
    for component in ["blank", "die", "binder", "punch"]:
        if component in op10:
            comp = op10[component]
            nodes = comp["node_displacement"].shape[0]

            if "node_displacement" in comp:
                disp_shape = comp["node_displacement"].shape
                timesteps = disp_shape[0] if len(disp_shape) == 3 else 1
                print(f"  {component}: {nodes} nodes, {timesteps} timesteps")
            else:
                print(f"  {component}: {nodes} nodes, no displacement")

    print("\n ----------------------- \n")

    # OP20: Springback operation
    print("OP20 - Cutting Process:")
    op20 = f["OP20"]
    print("  Components:", list(op20.keys()))

    # Analyze each component
    print("\nComponent Details:")
    for component in ["blank"]:
        if component in op20:
            comp = op20[component]
            nodes = comp["node_displacement"].shape[0]

            if "node_displacement" in comp:
                disp_shape = comp["node_displacement"].shape
                timesteps = disp_shape[0] if len(disp_shape) == 3 else 1
                print(f"  {component}: {nodes} nodes, {timesteps} timesteps")
            else:
                print(f"  {component}: {nodes} nodes, no displacement")

You can also automatically explore the file using [`display_structure`](https://ddacs.readthedocs.io/en/latest/api/utils/#ddacs.utils.display_structure)

In [ ]:
display_structure(h5_path)

## 4. Visualization Examples

The following sections demonstrate different visualization approaches for the simulation data.

In [ ]:
# Plot settings
FIGURE_SIZE = (8, 4)
FIGURE_DPI = 150
AXIS_LIMITS = [0, 110]

# Colors are set to the automatic colors of LS-Dyna
print(f"Component colors: {COMPONENT_COLORS}")

### 4.1. Visualize Point Cloud
Point clouds show the spatial distribution of nodes in each component. This visualization is useful for:
  - Understanding the overall geometry and tool positioning
  - Checking data quality and node density
  - Getting a quick overview of the forming setup at any timestep
  - Identifying areas with high/low mesh resolution

We use [`extract_point_cloud`](https://ddacs.readthedocs.io/en/latest/api/utils/#ddacs.utils.extract_point_cloud) to get coordinates and [`plot_point_cloud`](https://ddacs.readthedocs.io/en/latest/api/visualization/#ddacs.visualization.plot_point_cloud) for visualization.

Keep in mind, when working with different components, that a `-1` timestep index would result in different component states respectively (see previous cell descriptions). Therefore it is recommended to start from `0`.

In [ ]:
# Variables
TIMESTEP = 2  # The last timestep where tools and blank exist together

fig = plt.figure(figsize=FIGURE_SIZE, dpi=FIGURE_DPI)
ax = fig.add_subplot(111, projection="3d")

# Plot each component using ddacs.visualization
for component in COMPONENT_COLORS.keys():
    try:
        coords = extract_point_cloud(h5_path, component, timestep=TIMESTEP)
        alpha = 0.3 if component == "blank" else 0.7
        plot_point_cloud(
            coords,
            ax=ax,
            color=COMPONENT_COLORS[component],
            point_size=0.5,
            alpha=alpha,
            axis_limits=AXIS_LIMITS,
        )
        # Add to legend manually since plot_point_cloud doesn't add labels when using existing ax
        ax.scatter([], [], [], c=COMPONENT_COLORS[component], label=COMPONENT_NAMES[component])
        print(f"{component}: {coords.shape[0]} nodes")
    except Exception as e:
        print(f"Could not load {component}: {e}")

ax.set_title(f"Deep Drawing Setup - Simulation {sim_id} - Timestep {TIMESTEP} - OP10")
ax.legend(bbox_to_anchor=(1, 1), loc="upper left", fontsize="small")

plt.tight_layout()
plt.show()

### 4.2. Visualize Mesh
Mesh visualization shows the actual surface geometry with connected elements. Unlike point clouds, meshes reveal:
  - The continuous surface shape of components
  - Element connectivity and mesh quality
  - Realistic representation of tool and workpiece surfaces
  - Better understanding of contact interfaces between components

We use [`extract_mesh`](https://ddacs.readthedocs.io/en/latest/api/utils/#ddacs.utils.extract_mesh) to get vertices and triangles, and [`plot_mesh`](https://ddacs.readthedocs.io/en/latest/api/visualization/#ddacs.visualization.plot_mesh) for visualization.

In [ ]:
# Variables
TIMESTEP = 2  # The last timestep where tools and blank exist together
COMPONENT = "blank"  # Can be blank, die, punch or binder

# Extract mesh data
vertices, triangles = extract_mesh(h5_path, COMPONENT, timestep=TIMESTEP)
print(f"Mesh: {vertices.shape[0]} vertices, {triangles.shape[0]} faces")

# Plot using ddacs.visualization
ax = plot_mesh(
    vertices, triangles,
    color=COMPONENT_COLORS[COMPONENT],
    figsize=FIGURE_SIZE,
    title=f"Blank Mesh - Simulation {sim_id} - Timestep {TIMESTEP} - OP10",
    axis_limits=AXIS_LIMITS,
)

plt.tight_layout()
plt.show()

### 4.3. Visualize Mesh of Operation 20
Operation 20 (`OP20`) represents the cutting/trimming phase after forming. Key differences from `OP10`:
  - **Purpose**: Analyzes springback behavior during cutting operations
  - **Components**: Only the blank exists (tools are removed)
  - **Timesteps**: Fewer timesteps (0: state after cutting and before springback, 1: after springback)
  - **Physics**: Focuses on material recovery when constraints are released

Both previous visualization methods can be executed for OP20, with the limitations, that time steps differ and only `blank` component exist. As only the `blank` component exists, the negative timestep index `-1` does not cause any trouble.

In [ ]:
# Variables
TIMESTEP = -1  # The last timestep of the cutting operation
OPERATION = 20  # 20 for cutting operation, 10 for forming
COMPONENT = "blank"  # Only blank available in OP20

# Extract mesh data
vertices, triangles = extract_mesh(h5_path, COMPONENT, timestep=TIMESTEP, operation=OPERATION)
print(f"Mesh: {vertices.shape[0]} vertices, {triangles.shape[0]} faces")

# Plot using ddacs.visualization
ax = plot_mesh(
    vertices, triangles,
    color=COMPONENT_COLORS[COMPONENT],
    figsize=FIGURE_SIZE,
    title=f"Blank Mesh - Simulation {sim_id} - Timestep {TIMESTEP} - OP{OPERATION}",
    axis_limits=AXIS_LIMITS,
)

plt.tight_layout()
plt.show()

## 5. Visualize Physical Features
The following sections demonstrate how to visualize important physical quantities that reveal the manufacturing process physics. Examples are:
  - Material behavior during forming
  - Springback and dimensional accuracy

### 5.1 Thickness Distribution
Material thickness changes during deep drawing due to stretching and compression. The Thickness analysis reveals:
  - **Thinning zones** (Purple): High-stretch areas prone to necking or tearing
  - **Thickening zones** (Yellow): Compression areas, often near binder contact
  - **Critical areas**: Where thickness reduction exceeds material limits
  - **Process optimization**: Identifying parameter adjustments needed

In [ ]:
# Variables
TIMESTEP = -1  # The last timestep of the operation
OPERATION = 10  # Can be 10 or 20

# Extract mesh and thickness data
vertices, triangles = extract_mesh(h5_path, "blank", timestep=TIMESTEP, operation=OPERATION)
thickness = extract_element_thickness(h5_path, timestep=TIMESTEP, operation=OPERATION)

print(f"Thickness range: {thickness.min():.4f} - {thickness.max():.4f} mm")

# Global limits for comparable visualizations across simulations
THICKNESS_MIN = 0.8
THICKNESS_MAX = 1.15

# Plot using ddacs.visualization with thickness coloring
ax, cbar = plot_mesh(
    vertices, triangles,
    values=thickness,
    cmap="viridis",
    vmin=THICKNESS_MIN,
    vmax=THICKNESS_MAX,
    colorbar_label="Thickness [mm]",
    figsize=FIGURE_SIZE,
    title=f"Thickness Distribution - Simulation {sim_id} - Timestep {TIMESTEP} - OP{OPERATION}",
    axis_limits=AXIS_LIMITS,
)

plt.tight_layout()
plt.show()

### 5.2 Display Springback
Springback is the elastic recovery of material after tool removal. This phenomenon:
  - **Affects dimensional accuracy** of the final part
  - **Varies by location** depending on stress distribution during forming
  - **Influences tooling design** requiring compensation strategies
  - **Depends on material properties** and process parameters

The visualization interpretation:
  - **Color intensity**: Magnitude of springback displacement
  - **Hot spots** (red/yellow): Areas with significant dimensional changes
  - **Cool areas** (blue/purple): Stable regions with minimal springback
  - **Patterns**: Often concentrated near corners and high-curvature areas

The different visualization techniques are executed on the same simulation & operation. As the springback is always the last timestep and the timestep before the last timestep, the method [`extract_point_springback`](https://ddacs.readthedocs.io/en/latest/api/utils/#ddacs.utils.extract_point_springback) only requires the operation and simulation.

In [ ]:
# Variables
OPERATION = 20  # Can be 10 or 20.

# Get springback information
final_coords, displacement_vectors = extract_point_springback(
    h5_path, operation=OPERATION
)

# Calculate displacement magnitude for color mapping
displacement_magnitude = np.linalg.norm(displacement_vectors, axis=1)

# Settings for plotting.
global_springback_min = 0.0
global_springback_max = 1.4

# Statistics
print(f"Springback statistics:")
print(f" - Mean magnitude: {displacement_magnitude.mean():.4f} mm")
print(f" - Max magnitude: {displacement_magnitude.max():.4f} mm")
print(f" - Min magnitude: {displacement_magnitude.min():.4f} mm")
print(f" - Std deviation: {displacement_magnitude.std():.4f} mm")

#### 5.2.1 3D Point Cloud with Springback Magnitude
Shows the final geometry as a colored point cloud where colors represent springback magnitude     

In [ ]:
# Plot 3D point cloud colored by springback magnitude using ddacs.visualization
ax, cbar = plot_point_cloud(
    final_coords,
    values=displacement_magnitude,
    cmap="plasma",
    vmin=global_springback_min,
    vmax=global_springback_max,
    colorbar_label="Springback Magnitude [mm]",
    figsize=FIGURE_SIZE,
    title=f"Springback Point Cloud - Simulation {sim_id} - OP{OPERATION}",
    axis_limits=AXIS_LIMITS,
    point_size=1,
    alpha=0.8,
)

plt.show()

 #### 5.2.2 2D Top View (X-Y Projection)
The 2D projection using [`plot_2d_projection`](https://ddacs.readthedocs.io/en/latest/api/visualization/#ddacs.visualization.plot_2d_projection) provides a limited overview of springback patterns by removing the depth dimension. This approach has **limited resolution** as it:
  - **Loses Z-direction information**: Vertical springback components are not clearly visible
  - **Overlaps nodes**: Multiple Z-levels project to the same X-Y position
  - **Reduces spatial understanding**: Difficult to correlate with 3D geometry
  - **Masks complex patterns**: 3D springback vectors appear simplified

In [ ]:
# Plot 2D top view (X-Y projection) using ddacs.visualization
ax, cbar = plot_2d_projection(
    final_coords,
    values=displacement_magnitude,
    projection="xy",
    cmap="plasma",
    vmin=global_springback_min,
    vmax=global_springback_max,
    colorbar_label="Springback Magnitude [mm]",
    figsize=FIGURE_SIZE,
    title=f"Springback (Top View) - Simulation {sim_id} - OP{OPERATION}",
    point_size=2,
    alpha=0.8,
)

plt.tight_layout()
plt.show()

#### 5.2.3 Springback Vector Field (Quiver Plot)
Vector visualizations using [`plot_vectors`](https://ddacs.readthedocs.io/en/latest/api/visualization/#ddacs.visualization.plot_vectors) show both magnitude and direction of springback displacement:
  - **Arrow direction**: Shows the direction of material movement during springback
  - **Arrow length**: Proportional to displacement magnitude (specified by `SCALING`)
  - **Arrow density**: Controlled by subsampling (every Nth point `STEP`)
  - **Physical insight**: Reveals how material "flows back" after tool removal

**Interpretation tips:**
  - Arrows pointing outward indicate material expansion/recovery
  - Arrows pointing inward suggest local compression effects
  - Clustered arrows show regions of coordinated springback movement
  - Arrow length patterns reveal springback intensity gradients

In [ ]:
# Variables
STEP = 25  # Subsample every Nth point for arrow density
SCALE = 10  # Scaling factor for arrow length

# Plot vector field with springback magnitude coloring
ax, cbar = plot_vectors(
    final_coords,
    displacement_vectors,
    values=displacement_magnitude,
    step=STEP,
    scale=SCALE,
    arrow_color="darkred",
    arrow_alpha=0.8,
    cmap="plasma",
    vmin=global_springback_min,
    vmax=global_springback_max,
    colorbar_label="Springback Magnitude [mm]",
    point_size=1,
    point_alpha=0.6,
    figsize=FIGURE_SIZE,
    title=f"Springback: Points & Vectors - Simulation {sim_id} - OP{OPERATION}",
    axis_limits=AXIS_LIMITS,
)

plt.show()

## 6. PyTorch Integration (Optional)

This section demonstrates how to use the DDACS dataset with PyTorch's DataLoader for machine learning workflows.

**Requirements:** This section requires PyTorch. Install it first following the [official instructions](https://pytorch.org/get-started/locally/) for your CUDA version:

```bash
# Example for CPU-only
pip install torch

# For CUDA versions, see https://pytorch.org/get-started/locally/
```

The [`DDACSDataset`](https://ddacs.readthedocs.io/en/latest/api/pytorch/) class provides a PyTorch-compatible Dataset that can be used with DataLoader for batched iteration.

In [ ]:
try:
    from ddacs.pytorch import DDACSDataset
    from torch.utils.data import DataLoader
except ImportError:
    raise ImportError(
        "PyTorch is required for this section. "
        "Install it from https://pytorch.org/get-started/locally/"
    )

# Create dataset
dataset = DDACSDataset(data_dir)
print(f"Dataset size: {len(dataset)} simulations")

# Create DataLoader for batched iteration
dataloader = DataLoader(dataset, batch_size=4, shuffle=True)

# Iterate over batches
for batch_idx, (sim_ids, metadata_batch, h5_paths) in enumerate(dataloader):
    print(f"\nBatch {batch_idx}:")
    print(f"  Simulation IDs: {sim_ids.tolist()}")
    print(f"  Metadata shape: {metadata_batch.shape}")
    print(f"  H5 paths: {len(h5_paths)} files")
    
    if batch_idx >= 1:  # Show first 2 batches
        break